# Segmented-only augmentation and nested forest retuning

**Training data: 2,170 original comparisons + 15 segmented single centerlines. No paired-unit additions.** The full centroid shift remains the training target; peak timing is a transfer diagnostic, not a separately trained response.

The geometric definitions and primary shared 0.5% response window are unchanged from the extension audit. Three size options are tested: current $S_L$, no size, and edited-footprint $S_{\mathrm{edit}}$. All other six inputs stay unchanged. The revised footprint uses parent containing-domain probabilities and selected parent unit membership, excluding connectors for new composite cases.

## Two stages

1. **Fixed controls:** retrain the existing forest configuration with segmented-only additions for each size option. Current-size predictions are reused from the verified preceding experiment; the other two options are newly fitted across all 15 holdouts.
2. **Nested retuning:** eight forest configurations × three size options = 24 candidates. Each of 15 outer folds holds out an entire network. Three grouped inner folds select size and hyperparameters without access to that outer network. We also evaluate the inner-selected setting separately for each size option. The jointly selected result is the main evaluation of the complete selection procedure; choosing a size retrospectively from outer scores is exploratory.

The candidate set includes the exact frozen 500-tree absolute-error forest and seven 200-tree alternatives. Alternatives vary absolute/squared-error splitting, leaf size 1/2/5, split minimum 2/10, depth 10/16/unlimited, and feature sampling .75/1.0, with no pruning. This targets the observed inability to separate sparse whole-network cases; it is not an exhaustive hyperparameter search.

## Tuning objective and weighting

For each cohort, calculate $100\,|\hat y-y|/\max(|y|,1\mathrm{\ minute})$, average within each network, then across networks. The selection objective is the arithmetic mean of the original-cohort and segmented-cohort scores (50% each). This prevents either the 2,170 original records or the large centerline responses from silently dominating selection. It is a floored relative error, not ordinary MAPE or a confidence interval.

Training sample weights remain inverse network sample count, so each training network has equal total weight. Cohort balancing applies to **selection**, not training weights. Preprocessing and target standardization are trained within each fitting fold. Settings and size are selected jointly using only inner predictions. A final separate research artifact is selected using grouped CV on all data; its selection score is not an independent test score. The frozen model is never overwritten.

This remains exploratory after prior inspection of these networks. Only 15 independent networks and 15 segmented outcomes are available; group holdouts address shared-parent dependence, but do not create an untouched external test set.


## Completed results

The six paired units were excluded throughout. All 15 outer holdouts and the final all-data selection completed, with 24 candidates per tuning exercise. Three focused tests passed; all prediction joins are unique and complete (2,185 records per variant); serialization reproduced predictions to numerical tolerance.

| Experiment | Original R² | Original relative error | Segmented R² | Segmented relative error |
|---|---:|---:|---:|---:|
| Original training reference | 0.615 | 63.74% | −0.818 | 65.20% |
| Segmented additions, current size, fixed settings | −25.305 | 1228.77% | 0.222 | 48.11% |
| Segmented additions, no size, fixed settings | −0.037 | 101.01% | −0.612 | 88.73% |
| Segmented additions, edited size, fixed settings | 0.447 | 68.22% | 0.079 | 45.76% |
| Jointly retuned size + settings (nested evaluation) | 0.284 | 71.14% | −0.041 | 59.85% |

All outer folds selected edited size. Eight chose the original forest settings with that size, four chose 200-tree absolute-error forests with leaf=2, split=2, depth=16 and no pruning, and three chose the 200-tree squared-error configuration described below. The nested procedure did not improve generalization relative to fixed edited-size settings. The originally frozen reference settings were selected using these networks previously; they are a historical benchmark, not a new nested estimate.

The errors are not uniformly in one direction: segmented 22 has observed −5.263 h and predicted −13.638 h; 46 has −23.012 h and −5.591 h; 55 has −17.279 h and −5.592 h. Those are outer-held-out predictions. The current predictor/configuration family still does not reliably explain these large-collapse responses. This does not establish that segmented cases are invalid or should be discarded.

**Keep the frozen model unchanged.** Retain the segmented cases as an active investigation. A useful next analysis is whether geometrically available information about unchanged routing sections or spatial arrangement is missing from the current seven aggregated predictors. This experiment does not establish that hypothesis: training-weight choices, sparse large examples and forest limitations remain possible contributors. No additional predictor has been introduced here.

## Separate all-data research artifact

Grouped all-data selection chooses edited size with 200 trees, squared-error splitting, depth 10, minimum leaf size 5, minimum split size 10, feature fraction 0.75, bootstrap enabled and no pruning. Its inner selection errors are 66.21% original and 40.06% segmented; **these are selection scores, not test scores**. The nested scores above evaluate the selection procedure, not that one configuration independently.

The fitted research model is saved separately under the experiment output directory as `research_model.joblib`, with settings in `research_model.json`. Prediction must apply the edited-size transformation described in the script; do not supply the old containing-domain size in its place.


In [3]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'RAPID').is_dir())
OUT=ROOT/'RAPID/outputs/hydrograph_closed_unit_extension_20261019/segmented_search'
scores=pd.read_csv(OUT/'scores.csv')
main=scores.query("target=='centroid_shift_hours' and subset=='all'")
display(main[['variant','cohort','n','r2','mae','relative_error_pct']].round(3))

,variant,cohort,n,r2,mae,relative_error_pct
0,fixed_current,original,2170,-25.305,16.259,1228.771
6,fixed_current,segmented,15,0.222,212.952,48.109
12,fixed_edited,original,2170,0.447,3.055,68.222
18,fixed_edited,segmented,15,0.079,219.346,45.761
24,fixed_none,original,2170,-0.037,4.615,101.013
30,fixed_none,segmented,15,-0.612,332.842,88.734
36,original_training_reference,original,2170,0.615,2.823,63.741
42,original_training_reference,segmented,15,-0.818,342.740,65.198
48,tuned_current,original,2170,-25.245,16.150,1232.011
54,tuned_current,segmented,15,0.223,195.874,50.170


In [4]:
predictions=pd.read_parquet(OUT/'predictions.parquet')
z=predictions.query("cohort=='segmented'")
display(z.pivot(index='example',columns='variant',values='predicted_hours').join(
    z.drop_duplicates('example').set_index('example').centroid_shift_hours.rename('observed')).round(3))
p=predictions.query("variant in ['original_training_reference','fixed_current','tuned_joint'] and cohort=='segmented'")
fig=px.scatter(p,x='centroid_shift_hours',y='predicted_hours',color='variant',hover_data=['example'],
 title='Held-network segmented predictions',labels={'centroid_shift_hours':'Observed centroid shift (hours)','predicted_hours':'Predicted centroid shift (hours)'})
a=min(p.centroid_shift_hours.min(),p.predicted_hours.min());b=max(p.centroid_shift_hours.max(),p.predicted_hours.max())
fig.add_shape(type='line',x0=a,y0=a,x1=b,y1=b,line=dict(dash='dash',color='grey'))
fig.show()

,fixed_current,fixed_edited,fixed_none,original_training_reference,tuned_current,tuned_edited,tuned_joint,tuned_none,observed
example,,,,,,,,,
02,-1.013,-0.856,-4.295,-1.030,-1.586,-1.310,-1.310,-4.295,-0.966
03,-1.272,-1.244,-3.490,-1.266,-1.905,-1.244,-1.244,-3.490,-2.179
05,-2.008,-2.549,-2.344,-1.277,-2.245,-2.549,-2.549,-2.344,-6.535
13,-7.279,-5.465,-2.338,-1.233,-8.245,-5.465,-5.465,-2.338,-6.980
15,-3.333,-4.949,-1.570,-1.441,-5.049,-6.366,-6.366,-1.570,-4.460
21,-2.196,-2.163,-1.351,-1.264,-1.087,-1.953,-1.953,-1.351,-1.728
22,-9.111,-6.699,-1.507,-1.579,-9.111,-13.638,-13.638,-1.507,-5.263
37,-2.719,-4.337,-1.424,-1.381,-6.464,-4.674,-4.674,-1.424,-7.764
41,-8.100,-6.288,-1.063,-1.220,-7.865,-6.288,-6.288,-1.063,-13.244


In [5]:
network=pd.read_csv(OUT/'network_scores.csv',dtype={'example':str})
display(network.query("target=='centroid_shift_hours' and cohort=='original'").pivot(index='example',columns='variant',values='mae_minutes').round(3))
selected=pd.read_json(OUT/'outer_selections.json')
display(selected[selected.selection.eq('joint')][['held_example','index','selection_score','candidate']])
print('Separate all-data research configuration:')
print(json.dumps(json.loads((OUT/'research_model.json').read_text())['selected'],indent=2))

Separate all-data research configuration:
{
  "fingerprint": "9834e67346594fa7215c99f0edb77c5582d9719bcd6de0f81138d73e87d69266",
  "index": 23,
  "candidate": {
    "size": "edited",
    "params": {
      "n_estimators": 200,
      "criterion": "squared_error",
      "max_depth": 10,
      "min_samples_leaf": 5,
      "min_samples_split": 10,
      "max_features": 0.75,
      "bootstrap": true,
      "max_samples": null,
      "ccp_alpha": 0.0
    }
  },
  "selection_score": 53.13494555260487,
  "cohort_scores": {
    "original": 66.20623964846641,
    "segmented": 40.06365145674333
  }
}


variant,fixed_current,fixed_edited,fixed_none,original_training_reference,tuned_current,tuned_edited,tuned_joint,tuned_none
example,,,,,,,,
02,1.632,1.719,3.368,1.271,2.155,1.760,1.760,3.368
03,1.603,1.923,2.549,1.430,1.051,1.923,1.923,2.549
05,5.166,5.377,6.099,4.935,4.601,5.377,5.377,6.099
13,2.467,2.303,4.236,2.399,2.569,2.303,2.303,4.236
15,1.945,1.980,2.291,1.914,1.672,1.815,1.815,2.291
21,2.949,3.256,3.384,3.347,3.467,3.583,3.583,3.384
22,201.336,4.726,2.038,1.726,201.336,4.386,4.386,2.038
37,2.810,2.621,3.149,2.866,2.384,2.549,2.549,3.149
41,3.513,3.628,4.366,3.484,3.725,3.628,3.628,4.366


,held_example,index,selection_score,candidate
3,2,23,54.412304,"{'size': 'edited', 'params': {'n_estimators': ..."
7,3,16,56.418540,"{'size': 'edited', 'params': {'n_estimators': ..."
11,5,16,58.110356,"{'size': 'edited', 'params': {'n_estimators': ..."
15,13,16,62.364323,"{'size': 'edited', 'params': {'n_estimators': ..."
19,15,18,58.633173,"{'size': 'edited', 'params': {'n_estimators': ..."
23,21,18,52.344333,"{'size': 'edited', 'params': {'n_estimators': ..."
27,22,18,57.987611,"{'size': 'edited', 'params': {'n_estimators': ..."
31,37,18,55.028643,"{'size': 'edited', 'params': {'n_estimators': ..."
35,41,16,64.333905,"{'size': 'edited', 'params': {'n_estimators': ..."
39,45,16,66.539297,"{'size': 'edited', 'params': {'n_estimators': ..."


Scores excluding 22 or 54 are evaluation-only sensitivities, not refits without those networks. Peak-transfer scores are available in the same table. No pair outcomes contribute to fitting, selection, or evaluation.

Reproduce/resume with the project environment:

```bash
OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 MKL_NUM_THREADS=1 conda run --no-capture-output -n river-hierarchy-rivgraph python RAPID/investigations/closed_unit_extension_20261019/segmented_search.py --run --jobs 4
```

Source hashes, objective, candidate settings and exclusions are saved in `design.json`. Checkpoints are fingerprinted. Focused tests check equal-cohort/equal-network scoring, size-only transformation and inclusion of the exact original forest configuration. Final serialization is checked to numerical tolerance.

## Routing and predictor follow-up

See [segmented diagnostics and example 22](hydrograph_segmented_diagnostics.ipynb). The −5.263-hour centroid shift is reproduced and changes by only 0.192 seconds under joint parent/child timestep and cell refinement. Its 5 m/s cap substantially suppresses the advance: isolated sensitivities give −7.855 h at 10 m/s and −8.243 h with no active upper cap. The unchanged reaches cancel in the route-time difference; the 14 collapsed blocks account for it. Large cross-network differences in effective speeds are absent from the current geometric predictor set. These are model diagnostics, not new predictor inputs or changes to the final calibrated simulations.